In [1]:
from google.colab import drive # remove the cell if not using colab
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
base_path = Path('/content/drive/MyDrive/solvro_dane') # change path here!

# Klasyfikacja pasażerów Titanica
Nie wiemy czy dla DiCaprio było miejsce na drzwiach, ale wiemy że grdyby był tam Wojfer87 to by z nimi wyciskał pompki na górze lodowej. Teraz twoja pora na wyciskanie.
#Twoje zadnie to:
**stworzenie modelu przewidującego szanse przeżycia katastrofy Titanica**.

![https://i1.jbzd.com.pl/contents/2025/11/normal/v5Fth4DcPpPPxSrXQ5rCbAgZ8EifWiiF.png](https://i1.jbzd.com.pl/contents/2025/11/normal/v5Fth4DcPpPPxSrXQ5rCbAgZ8EifWiiF.png "Wojfer")



#### Twoim celem będzie jest wytrenowanie modeli do klasyfikacji każdego pasażera Titanica jako ofiary (0) lub osoby, która przeżyła (1).

Poniżej znajdziesz pytania, które mogą być pomocne w zadaniu:

- Czego nauczyło Cię o badanym zbiorze danych poprzednie zadanie? Jak możesz wykorzystać wyciągnięte z niego wnioski w procesie tworzenia modelu?
- Jak przeprowadzenie standaryzacji danych może wpływać na zachowanie modelu?
- Co mój model robi i w jaki sposób?
- Jak nie przetrenować wybranego modelu?
- Jaki wynik klasyfikacji możemy uznać za *dobry*?


Wymagania:
- Wypisz obserwacje z pierwszego zadania, które pomogą Ci w tym. Co było przydatne, a co okazało się bezużyteczne?
- [Nie doprowadź](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) do ~~przecieku statku~~ wycieku danych (np. nie ucz modelu na danych testowych). Nauczone modele odpal na danych treningowych i testowych - opisz uzyskane wyniki.
- Stwórz baseline, czyli dla porównania sprawdź jak z zadaniem radzi sobie [Dummy Classifier](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html) (jeśli Twój docelowy model radzi sobie gorzej - uciekaj)
- Przeprowadź badania na dwóch wybranych modelach uczenia maszynowego (np. spośród: drzew decyzyjnych, SVM, MLP, KNN, z gwiazdką [XGBoost](https://xgboost.readthedocs.io/en/stable))
- W badaniach użyj wybranych metryk. Wybór uzasadnij.
- Dla każdego modelu wybierz co najmniej dwa hiperparametry i przeprowadź badania zależności wyników metryk od wartości hiperparametrów. Zwizualizuj wszystko ładnie, zastanów się dlaczego tak mogło być i wyciągnij i wypisz wnioski.
- Podsumuj przeprowadzone badania, wypisz wnioski.

Niezmiennie, zadbaj o czytelność kodu i nazewnictwo zmiennych. Jeśli jakiś wycinek kodu się powtarza, to wyodrębnij go do funkcji. Postaraj się zamieszczać swoje wnioski w postaci komentarza `Markdown`.

Jeśli chcesz, możesz sprawdzić (przyjmując pewne założenia), jakie byłyby Twoje szanse na Titanicu.

Uwaga! Jeśli Titanic to dla Ciebie nic i baaaaardzo chcesz to możesz w ramach tego zadania zająć się [bardziej wymagającym](https://archive.ics.uci.edu/dataset/365/polish+companies+bankruptcy+data) zbiorem.

In [3]:
titanic_df = pd.read_csv(base_path / 'titanic_final.csv', index_col='PassengerId')

# Wnioski z EDA
Większą szansę na przeżycie miały:
* kobiety
* dzieci
* pasażerowie wyższych klas
* pasażerowie z przypisanymi kabinami (łączy się z klasą)

# Ładowanie i dzielenie danych

In [4]:
from sklearn.model_selection import train_test_split

X = titanic_df.drop(columns=['Survived'])
y = titanic_df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=.8, shuffle=True)

# Skalowanie danych


In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Baseline - Dummy Classifier

In [6]:
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, classification_report
from sklearn.dummy import DummyClassifier

model = DummyClassifier(strategy='most_frequent') # dane nie są zbalansowane - większość zginęła
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print(classification_report(y_test, y_pred))
confusion_matrix(y_test, y_pred)

              precision    recall  f1-score   support

           0       0.64      1.00      0.78       114
           1       0.00      0.00      0.00        65

    accuracy                           0.64       179
   macro avg       0.32      0.50      0.39       179
weighted avg       0.41      0.64      0.50       179



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


array([[114,   0],
       [ 65,   0]])

# Support Vector Machine

In [7]:
from sklearn.svm import SVC

svm = SVC(kernel='linear')  # liniowe SVM
svm.fit(X_train_scaled, y_train)

y_pred = svm.predict(X_test_scaled)

print(classification_report(y_test, y_pred))
confusion_matrix(y_test, y_pred)

              precision    recall  f1-score   support

           0       0.85      0.85      0.85       114
           1       0.74      0.74      0.74        65

    accuracy                           0.81       179
   macro avg       0.79      0.79      0.79       179
weighted avg       0.81      0.81      0.81       179



array([[97, 17],
       [17, 48]])

# Multi Layer Perceptron

In [8]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(max_iter=1000, random_state=42, hidden_layer_sizes=(64, 32), early_stopping=True)
mlp.fit(X_train_scaled, y_train)

y_pred = mlp.predict(X_test_scaled)

print(classification_report(y_test, y_pred))
confusion_matrix(y_test, y_pred)

              precision    recall  f1-score   support

           0       0.84      0.81      0.83       114
           1       0.69      0.74      0.71        65

    accuracy                           0.78       179
   macro avg       0.76      0.77      0.77       179
weighted avg       0.79      0.78      0.78       179



array([[92, 22],
       [17, 48]])

# K Nearest Neighbors


In [9]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=10, metric='cosine')
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)

print(classification_report(y_test, y_pred))
confusion_matrix(y_test, y_pred)

              precision    recall  f1-score   support

           0       0.82      0.88      0.85       114
           1       0.75      0.66      0.70        65

    accuracy                           0.80       179
   macro avg       0.79      0.77      0.78       179
weighted avg       0.80      0.80      0.80       179



array([[100,  14],
       [ 22,  43]])

W zależności od tego jak załadują się dane, modele dają znacząco różne wyniki.

# Cross-Validation

In [10]:
from sklearn.model_selection import cross_val_score

svm_score = cross_val_score(svm, X, y, cv=5, scoring='f1')
mlp_score = cross_val_score(mlp, X, y, cv=5, scoring='f1')
knn_score = cross_val_score(knn, X, y, cv=5, scoring='f1')

print('---------- SVM ----------')
print(svm_score)
print(f"srednia: {svm_score.mean():.3f}  odchylenie: {svm_score.std():.3f}")

print('\n---------- MLP ----------')
print(mlp_score)
print(f"srednia: {mlp_score.mean():.3f}  odchylenie: {mlp_score.std():.3f}")

print('\n---------- KNN ----------')
print(knn_score)
print(f"srednia: {knn_score.mean():.3f}  odchylenie: {knn_score.std():.3f}")

---------- SVM ----------
[0.7826087  0.76691729 0.75384615 0.70588235 0.82089552]
srednia: 0.766  odchylenie: 0.038

---------- MLP ----------
[0.38655462 0.55932203 0.50485437 0.59459459 0.56198347]
srednia: 0.521  odchylenie: 0.073

---------- KNN ----------
[0.58015267 0.64       0.59016393 0.61016949 0.60714286]
srednia: 0.606  odchylenie: 0.020


Wybrałem metrykę f1 - jest ona harmoniczną średnią z precision i recall. Model musi dobrze wybierać, ale też nie może się za często mylić. Z obecnymi parametrami dość często się myli.

In [11]:
!pip install optuna
import optuna
from sklearn.pipeline import Pipeline
import optuna.visualization as vis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 20.6 MB/s eta 0:00:00


# Szukanie hiperparametrów dla SVM

In [12]:
def svm_search(trial):
  c_param = trial.suggest_float('C', 0.01, 100, log=True)

  kernel_param = trial.suggest_categorical('kernel', ['linear', 'rbf'])

  if kernel_param == 'rbf':
      gamma_param = trial.suggest_float('gamma', 0.0001, 10, log=True)
  else:
      gamma_param = 'scale'

  class_weight_param = trial.suggest_categorical('class_weight', [None, 'balanced'])

  svm = SVC(C=c_param, kernel=kernel_param, gamma=gamma_param,
            class_weight=class_weight_param, random_state=42)

  pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', svm)
    ])

  score = cross_val_score(pipeline, X, y, cv=5, scoring='f1').mean()
  return score

study = optuna.create_study(direction='maximize')

study.optimize(svm_search, n_trials=100)

print("\n--- ZAKOŃCZONO ---")
print(f"Najlepszy F1: {study.best_value:.4f}")
print("Najlepsze parametry:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")


[I 2026-08-17 10:13:56,856] A new study created in memory with name: no-name-915744a2-5ecc-42f3-9d4b-a85fcae1a622
[I 2026-08-17 10:13:57,325] Trial 0 finished with value: 0.7641497402364538 and parameters: {'C': 3.113956373389924, 'kernel': 'linear', 'class_weight': None}. Best is trial 0 with value: 0.7641497402364538.
[I 2026-08-17 10:13:57,641] Trial 1 finished with value: 0.719355243412165 and parameters: {'C': 0.24738394854481321, 'kernel': 'rbf', 'gamma': 0.002319773479267346, 'class_weight': None}. Best is trial 0 with value: 0.7641497402364538.
[I 2026-08-17 10:13:57,851] Trial 2 finished with value: 0.7548308255854559 and parameters: {'C': 0.47876771560482945, 'kernel': 'rbf', 'gamma': 0.4582955586642765, 'class_weight': None}. Best is trial 0 with value: 0.7641497402364538.
[I 2026-08-17 10:13:58,069] Trial 3 finished with value: 0.7288546142308023 and parameters: {'C': 0.024048144542257684, 'kernel': 'linear', 'class_weight': None}. Best is trial 0 with value: 0.764149740236


--- ZAKOŃCZONO ---
Najlepszy F1: 0.7701
Najlepsze parametry:
  C: 1.7470416553203578
  kernel: rbf
  gamma: 0.034098617721104195
  class_weight: None


In [13]:
study.trials_dataframe().sort_values(by='value', ascending=False).head(10)

,number,value,datetime_start,datetime_complete,duration,params_C,params_class_weight,params_gamma,params_kernel,state
58,58,0.770147,2026-08-17 10:14:52.175721,2026-08-17 10:14:52.342833,0 days 00:00:00.167112,1.747042,None,0.034099,rbf,COMPLETE
0,0,0.764150,2026-08-17 10:13:56.858909,2026-08-17 10:13:57.325474,0 days 00:00:00.466565,3.113956,None,NaN,linear,COMPLETE
16,16,0.764150,2026-08-17 10:14:19.040585,2026-08-17 10:14:20.830896,0 days 00:00:01.790311,7.202913,None,NaN,linear,COMPLETE
12,12,0.764150,2026-08-17 10:14:14.682700,2026-08-17 10:14:16.021280,0 days 00:00:01.338580,3.278229,None,NaN,linear,COMPLETE
28,28,0.764150,2026-08-17 10:14:38.017161,2026-08-17 10:14:38.268082,0 days 00:00:00.250921,2.021574,None,NaN,linear,COMPLETE
31,31,0.764150,2026-08-17 10:14:38.613404,2026-08-17 10:14:39.110358,0 days 00:00:00.496954,6.572142,None,NaN,linear,COMPLETE
33,33,0.764150,2026-08-17 10:14:39.721577,2026-08-17 10:14:40.129115,0 days 00:00:00.407538,4.603090,None,NaN,linear,COMPLETE
32,32,0.764150,2026-08-17 10:14:39.111666,2026-08-17 10:14:39.720385,0 days 00:00:00.608719,10.108106,None,NaN,linear,COMPLETE
25,25,0.764150,2026-08-17 10:14:36.754486,2026-08-17 10:14:37.146370,0 days 00:00:00.391884,3.359299,None,NaN,linear,COMPLETE
27,27,0.764150,2026-08-17 10:14:37.371239,2026-08-17 10:14:38.015945,0 days 00:00:00.644706,7.799716,None,NaN,linear,COMPLETE


In [14]:
vis.plot_optimization_history(study)

In [15]:
vis.plot_param_importances(study)

In [16]:
vis.plot_parallel_coordinate(study)

# Szukanie hiperparametrów dla MLP

In [17]:
def mlp_search(trial):
  layer_options = {
        '32': (32,),
        '32_16': (32, 16),
        '64': (64,),
        '64_32': (64, 32),
        '128_64': (128, 64),
        '128_64_32': (128, 64, 32)
    }

  layer_name = trial.suggest_categorical('hidden_layer_sizes', list(layer_options.keys()))
  hidden_layer_sizes = layer_options[layer_name]

  activation = trial.suggest_categorical('activation', ['relu', 'tanh'])

  alpha = trial.suggest_float('alpha', 0.00001, 0.1, log=True)
  learning_rate_init = trial.suggest_float('learning_rate_init', 0.0001, 0.1, log=True)

  mlp = MLPClassifier(hidden_layer_sizes=hidden_layer_sizes, activation=activation, alpha=alpha,
                      learning_rate_init=learning_rate_init, max_iter=1000, random_state=42, early_stopping=True)

  pipeline = Pipeline([('scaler', StandardScaler()), ('mlp', mlp)])

  score = cross_val_score(pipeline, X, y, cv=5, scoring='f1').mean()

  return score

study = optuna.create_study(direction='maximize')

study.optimize(mlp_search, n_trials=100)

print("\n--- ZAKOŃCZONO ---")
print(f"Najlepszy F1: {study.best_value:.4f}")
print("Najlepsze parametry:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-08-17 10:15:17,298] A new study created in memory with name: no-name-429e1d71-5904-4f9e-80ee-db7f21e6f83a
[I 2026-08-17 10:15:27,825] Trial 0 finished with value: 0.7287478302117025 and parameters: {'hidden_layer_sizes': '128_64', 'activation': 'tanh', 'alpha': 0.025313891276822816, 'learning_rate_init': 0.0062518888311978195}. Best is trial 0 with value: 0.7287478302117025.
[I 2026-08-17 10:15:33,880] Trial 1 finished with value: 0.7193090823562598 and parameters: {'hidden_layer_sizes': '128_64', 'activation': 'relu', 'alpha': 8.65722483047466e-05, 'learning_rate_init': 0.08026481112677171}. Best is trial 0 with value: 0.7287478302117025.
[I 2026-08-17 10:15:35,427] Trial 2 finished with value: 0.7495144214546577 and parameters: {'hidden_layer_sizes': '64', 'activation': 'relu', 'alpha': 0.0017314956540753208, 'learning_rate_init': 0.03607973706828513}. Best is trial 2 with value: 0.7495144214546577.
[I 2026-08-17 10:15:37,147] Trial 3 finished with value: 0.44884982924415506 


--- ZAKOŃCZONO ---
Najlepszy F1: 0.7746
Najlepsze parametry:
  hidden_layer_sizes: 128_64_32
  activation: relu
  alpha: 9.093391502124464e-05
  learning_rate_init: 0.0017130453864760537


In [18]:
study.trials_dataframe().sort_values(by='value', ascending=False).head(10)

,number,value,datetime_start,datetime_complete,duration,params_activation,params_alpha,params_hidden_layer_sizes,params_learning_rate_init,state
82,82,0.774609,2026-08-17 10:17:43.769802,2026-08-17 10:17:45.064805,0 days 00:00:01.295003,relu,0.000091,128_64_32,0.001713,COMPLETE
92,92,0.774586,2026-08-17 10:18:00.433411,2026-08-17 10:18:01.796179,0 days 00:00:01.362768,relu,0.000043,128_64_32,0.001675,COMPLETE
38,38,0.772751,2026-08-17 10:16:30.071060,2026-08-17 10:16:31.371806,0 days 00:00:01.300746,relu,0.000098,128_64_32,0.001699,COMPLETE
96,96,0.771518,2026-08-17 10:18:06.033196,2026-08-17 10:18:07.249487,0 days 00:00:01.216291,relu,0.000036,128_64_32,0.001997,COMPLETE
81,81,0.770632,2026-08-17 10:17:39.495671,2026-08-17 10:17:43.768120,0 days 00:00:04.272449,relu,0.000097,128_64_32,0.001606,COMPLETE
95,95,0.770079,2026-08-17 10:18:04.722800,2026-08-17 10:18:06.030180,0 days 00:00:01.307380,relu,0.000069,128_64_32,0.002018,COMPLETE
44,44,0.769725,2026-08-17 10:16:42.724104,2026-08-17 10:16:44.161653,0 days 00:00:01.437549,relu,0.000056,128_64_32,0.001738,COMPLETE
84,84,0.768506,2026-08-17 10:17:46.532570,2026-08-17 10:17:48.029823,0 days 00:00:01.497253,relu,0.000069,128_64_32,0.001582,COMPLETE
52,52,0.767805,2026-08-17 10:16:56.300795,2026-08-17 10:16:57.566070,0 days 00:00:01.265275,relu,0.000061,128_64_32,0.001858,COMPLETE
21,21,0.767770,2026-08-17 10:16:12.617334,2026-08-17 10:16:12.953583,0 days 00:00:00.336249,relu,0.001862,64,0.043486,COMPLETE


In [19]:
vis.plot_optimization_history(study)

In [20]:
vis.plot_param_importances(study)

In [21]:
vis.plot_parallel_coordinate(study)

# Szukanie hiperparametrów dla KNN

In [22]:
def knn_search(trial):
  n_neighbors = trial.suggest_int('n_neighbors', 1, 50)

  weights = trial.suggest_categorical('weights', ['uniform', 'distance'])

  metric = trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'cosine'])

  knn = KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights, metric=metric)

  pipeline = Pipeline([('scaler', StandardScaler()), ('knn', knn)])

  score = cross_val_score(pipeline, X, y, cv=5, scoring='f1').mean()
  return score

study = optuna.create_study(direction='maximize')

study.optimize(knn_search, n_trials=100)

print("\n--- ZAKOŃCZONO ---")
print(f"Najlepszy F1: {study.best_value:.4f}")
print("Najlepsze parametry:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")


[I 2026-08-17 10:18:15,502] A new study created in memory with name: no-name-2464fbb2-84b0-432c-a2b4-9c05972ce16d
[I 2026-08-17 10:18:15,605] Trial 0 finished with value: 0.7237116529107894 and parameters: {'n_neighbors': 33, 'weights': 'distance', 'metric': 'cosine'}. Best is trial 0 with value: 0.7237116529107894.
[I 2026-08-17 10:18:15,744] Trial 1 finished with value: 0.7210588429508726 and parameters: {'n_neighbors': 35, 'weights': 'uniform', 'metric': 'cosine'}. Best is trial 0 with value: 0.7237116529107894.
[I 2026-08-17 10:18:15,889] Trial 2 finished with value: 0.6979627530343911 and parameters: {'n_neighbors': 47, 'weights': 'uniform', 'metric': 'euclidean'}. Best is trial 0 with value: 0.7237116529107894.
[I 2026-08-17 10:18:15,966] Trial 3 finished with value: 0.7388167509436949 and parameters: {'n_neighbors': 41, 'weights': 'distance', 'metric': 'euclidean'}. Best is trial 3 with value: 0.7388167509436949.
[I 2026-08-17 10:18:16,071] Trial 4 finished with value: 0.7202632


--- ZAKOŃCZONO ---
Najlepszy F1: 0.7525
Najlepsze parametry:
  n_neighbors: 31
  weights: distance
  metric: euclidean


In [23]:
study.trials_dataframe().sort_values(by='value', ascending=False).head(10)

,number,value,datetime_start,datetime_complete,duration,params_metric,params_n_neighbors,params_weights,state
24,24,0.752546,2026-08-17 10:18:17.575247,2026-08-17 10:18:17.651632,0 days 00:00:00.076385,euclidean,31,distance,COMPLETE
32,32,0.752546,2026-08-17 10:18:18.304916,2026-08-17 10:18:18.379435,0 days 00:00:00.074519,euclidean,31,distance,COMPLETE
61,61,0.752546,2026-08-17 10:18:20.688421,2026-08-17 10:18:20.759852,0 days 00:00:00.071431,euclidean,31,distance,COMPLETE
54,54,0.752546,2026-08-17 10:18:20.136333,2026-08-17 10:18:20.208773,0 days 00:00:00.072440,euclidean,31,distance,COMPLETE
74,74,0.752546,2026-08-17 10:18:21.787959,2026-08-17 10:18:21.861725,0 days 00:00:00.073766,euclidean,31,distance,COMPLETE
76,76,0.752546,2026-08-17 10:18:21.938992,2026-08-17 10:18:22.028383,0 days 00:00:00.089391,euclidean,31,distance,COMPLETE
86,86,0.752546,2026-08-17 10:18:22.765299,2026-08-17 10:18:22.835068,0 days 00:00:00.069769,euclidean,31,distance,COMPLETE
93,93,0.752546,2026-08-17 10:18:23.304693,2026-08-17 10:18:23.376396,0 days 00:00:00.071703,euclidean,31,distance,COMPLETE
91,91,0.752546,2026-08-17 10:18:23.150698,2026-08-17 10:18:23.224525,0 days 00:00:00.073827,euclidean,31,distance,COMPLETE
52,52,0.752546,2026-08-17 10:18:19.984972,2026-08-17 10:18:20.057529,0 days 00:00:00.072557,euclidean,31,distance,COMPLETE


In [24]:
vis.plot_optimization_history(study)

In [25]:
vis.plot_param_importances(study)

In [26]:
vis.plot_parallel_coordinate(study)

# Moje szanse na tytaniku 💀

In [27]:
print(titanic_df.columns)
titanic_df.head()

Index(['Survived', 'Pclass', 'Sex', 'Age', 'Fare', 'FamilySize', 'IsAlone',
       'Embarked_Q', 'Embarked_S', 'Cabin_B', 'Cabin_C', 'Cabin_D', 'Cabin_E',
       'Cabin_F', 'Cabin_G', 'Cabin_T', 'Cabin_Unknown', 'Title_Miss',
       'Title_Mr', 'Title_Mrs', 'Title_Rare'],
      dtype='object')


,Survived,Pclass,Sex,Age,Fare,FamilySize,IsAlone,Embarked_Q,Embarked_S,Cabin_B,...,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Cabin_Unknown,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
PassengerId,,,,,,,,,,,,,,,,,,,,,
1,0,3,0,22.0,7.2500,2,0,0,1,0,...,0,0,0,0,0,1,0,1,0,0
2,1,1,1,38.0,71.2833,2,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3,1,3,1,26.0,7.9250,1,1,0,1,0,...,0,0,0,0,0,1,1,0,0,0
4,1,1,1,35.0,53.1000,2,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
5,0,3,0,35.0,8.0500,1,1,0,1,0,...,0,0,0,0,0,1,0,1,0,0


In [28]:
ja = pd.DataFrame([{
    'Pclass': 1,
    'Sex': 0,
    'Age': 21.0,
    'Fare': 53.0,
    'FamilySize': 2,
    'IsAlone': 0,
    'Embarked_Q': 1,
    'Embarked_S': 0,
    'Cabin_B': 0,
    'Cabin_C': 0,
    'Cabin_D': 0,
    'Cabin_E': 0,
    'Cabin_F': 0,
    'Cabin_G': 0,
    'Cabin_T': 0,
    'Cabin_Unknown': 0,
    'Title_Miss': 0,
    'Title_Mr': 1,
    'Title_Mrs': 0,
    'Title_Rare': 0
}])

# użyje dobrych parametrów
  # * wartości mogą się różnić bo raczej odpale cały plik jak skończę
svm = SVC(kernel='rbf', C=7.925165, gamma=0.012747, class_weight=None)
svm.fit(X_train_scaled, y_train)

mlp = MLPClassifier(max_iter=1000, random_state=42, hidden_layer_sizes=(128, 64, 32), alpha=0.000312, learning_rate_init=0.001695, early_stopping=True)
mlp.fit(X_train_scaled, y_train)

knn = KNeighborsClassifier(n_neighbors=31, metric='euclidean', weights='distance')
knn.fit(X_train_scaled, y_train)

ja_scaled = scaler.transform(ja)

svm_pred = svm.predict(ja_scaled)
mlp_pred = mlp.predict(ja_scaled)
knn_pred = knn.predict(ja_scaled)

for pred in [svm_pred, mlp_pred, knn_pred]:
  if pred[0] == 1:
    print('Żyje!!!')
  else:
    print('GG')

GG
Żyje!!!
GG


# Wnioski
* niestety wygląda na to, że nie przeżyłbym na tytaniku, a przynajmniej 2 na 3 modele tak zdecydowały (mimo, że dałem sobie 1 klasę)
* wszystkie 3 modele pokonały model bazowy
* wszystkie 3 modele osiągnęły bardzo podobny poziom
* poprawność przewidywania jest znacząco zależna od tego jakie dane wylosują się jako treningowe